# Demo ba nhiệm vụ của đồ án

Chọn **Restart Kernel and Run All Cells** với kernel Python của `myenv` trong workspace đã có dữ liệu và checkpoint thực nghiệm. Mọi phép suy luận trong notebook chạy trên CPU. Khi vấn đáp có thể chạy trước, rồi mở từng mục bằng mục lục Jupyter.

1. **VG150** — dự đoán quan hệ và kiểm tra 5 Hit, 5 Miss của phụ lục.
2. **GQA (Task A)** — so hai mô hình trên 10 ca của bảng GQA. **MagicBrush và He (Task B)** — xem câu lệnh biến thành thao tác chỉnh sửa thế nào.
3. **VN100** — so đồ thị của hai ảnh, tính điểm chỉnh sửa và bảo toàn rồi đối chiếu nhãn tay.

Các ca được chọn để giải thích kết quả ở cấp ví dụ. Số liệu toàn tập nằm trong báo cáo.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tools/graph_editing_live_demo_v1.py").is_file())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [ ]:
import pandas as pd
import torch
from IPython.display import HTML, Image, Markdown, display
from src.datasets.vg150 import VG150AnnotationLoader, VG150GraphDataset
from src.features.visual import GeometryVisualFeatureExtractor, VisualFeatureExtractor
from src.graph.builder import SceneGraphBuilder
from src.models.gat import build_gat_model
from src.editing.operations import EditGraph
from src.evaluation.reference_predcls import evaluate_reference_image
from src.knowledge.transe.core import sha256
from tools.graph_editing_live_demo_v1 import predict
from tools.level3_submission_v1.demo import graph_html, image_pair_html, open_bundle, recompute, table, task2_demo

torch.set_num_threads(2)

## 1. VG150 · Quan hệ nào nối hai đối tượng?

Hộp và nhãn đối tượng được cho sẵn. GAT+CLIP (seed 42) dự đoán quan hệ có hướng cho từng cặp. Ở chế độ **graph constraint (GC)**, mỗi cặp chỉ giữ một quan hệ. Evaluator xét 50 bộ ba đứng đầu mỗi ảnh và đối chiếu predicate, lớp đối tượng cùng độ giao nhau của hộp với từng chú thích chuẩn.

Mười dòng bên dưới là 10 chú thích trên 5 ảnh validation ở bảng VG150 của phụ lục. Hộp đỏ đánh dấu chủ thể, hộp xanh đánh dấu đối tượng.

In [ ]:
CASE_ROOT = ROOT / 'reports/level1/submission/cases'
vg_manifest = json.loads((CASE_ROOT / 'cases.json').read_text())
checkpoint_path = ROOT / vg_manifest['checkpoint']
assert sha256(checkpoint_path) == vg_manifest['checkpoint_sha256']
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = checkpoint['config']
DATA = ROOT / 'data/raw/vg150'
for filename, digest in config['data_sha256'].items():
    assert sha256(DATA / filename) == digest
CACHE = ROOT / 'data/cache/vg150-clip-vit-b32-v1'
assert sha256(CACHE / 'manifest.json') == config['visual_cache_manifest_sha256']
loader = VG150AnnotationLoader(root=DATA, split='val', split_manifest=DATA / 'splits.json',
                              box_policy=config['box_policy'])
features = GeometryVisualFeatureExtractor(VisualFeatureExtractor(CACHE, DATA, config['box_policy']))
dataset = VG150GraphDataset(loader=loader, feature_extractor=features, graph_builder=SceneGraphBuilder())
model = build_gat_model(config)
model.load_state_dict(checkpoint['model'])
model.eval()
vocab = json.loads((DATA / 'VG-SGG-dicts-with-attri.json').read_text())
vg_predicates = {v:k for k,v in vocab['predicate_to_idx'].items()}

In [ ]:
def vg_case(case):
    graph = dataset[case['validation_index']]
    assert str(graph.image_id) == case['image_id']
    with torch.inference_mode():
        logits = model(x=graph.x, edge_label_index=graph.edge_label_index,
                       edge_index=graph.edge_index, object_labels=graph.object_labels)
    scores, labels = logits.softmax(-1)[:, 1:].max(-1)
    pairs = graph.edge_label_index
    ranks = scores.sort(descending=True).indices.argsort() + 1
    pair, = torch.where((pairs[0] == case['subject_node']) & (pairs[1] == case['object_node']))[0]
    pred, rank = vg_predicates[int(labels[pair]) + 1], int(ranks[pair])
    row = case['gt_annotation_row']
    gt = vg_predicates[int(graph.gt_predicate[row])]
    assert graph.gt_relation_index[:, row].tolist() == [case['subject_node'], case['object_node']]
    result = evaluate_reference_image(logits, pairs, graph.gt_relation_index[:, row:row+1],
                                      graph.gt_predicate[row:row+1], graph.boxes, graph.object_labels, ks=(50,))
    kind = 'hit' if result['hits'][50] else 'miss'
    assert (gt, pred, rank, kind) == (case['gt_predicate'], case['same_pair_prediction'],
                                      case['same_pair_rank'], case['kind'])
    return [case['case_id'], graph.image_id, case['subject_label'], case['object_label'], gt, pred, rank]

for kind in ['hit', 'miss']:
    selected = vg_manifest['cases'][kind]
    assert len(selected) == 5
    display(HTML(table(['Ca', 'Ảnh', 'Chủ thể', 'Đối tượng', 'Nhãn chuẩn', 'GAT', 'Hạng cặp'],
                       [vg_case(case) for case in selected])))
    display(Image(filename=str(CASE_ROOT / f'{kind}-contact-sheet.png'), width=950))

Đọc Hit 3 và Miss 3 trên ảnh 89: `flower → in → vase` đứng hạng 1, còn `shelf → on → table` đứng hạng 64. Cả hai đều đúng nhãn, chỉ bộ ba đầu nằm trong top-50. Ở Miss 5, cùng một cặp có hai nhãn chuẩn; GC giữ `with`, nên chú thích `has` bị bỏ sót.

## 2A. GQA · Hai mô hình xếp hạng cùng cặp thế nào?

Dùng hai checkpoint PE-PLPR và PE-PLPR + AlignG (seed 42). Ô bên dưới duyệt ảnh validation theo thứ tự manifest, chọn đúng quy tắc của bảng GQA trong phụ lục và kiểm tra Hit/Miss bằng evaluator IoU. Mỗi nhóm có năm quan hệ chuẩn khác nhau.

In [ ]:
from tools.gqa_test_v1.runner import build, read
from experiments.aligng_fast_v1.batching import collate, model_inputs
from src.datasets.gqa_full import GQAFullDataset

manifest_path = Path('data/processed/level2/gqa-v1/manifest.json')
manifest = read(manifest_path)
vocab = manifest['vocabulary']
predicates = {i: p for p, i in vocab['predicate_to_idx'].items()}
objects = {i: p for p, i in vocab['label_to_idx'].items()}
directory = Path('data/cache/gqa-full-clip-v1')
metadata = read(directory / 'manifest.json')
assert metadata['identity']['manifest_sha256'] == sha256(manifest_path)
graphs = GQAFullDataset(directory, metadata, 'val')
assert graphs.ids == manifest['splits']['val']
jobs = read(Path('reports/level2/gqa-test-v1/training-audit.json'))['jobs']
def checkpoint(model):
    job, = [j for j in jobs if j['model'] == model and j['seed'] == 42]
    return build(job)  # kiểm SHA-256 và đọc checkpoint trên CPU

aligng, dims = checkpoint('pe-plpr-aligng')
control, _ = checkpoint('pe-plpr')

In [ ]:
def pair_predictions(logits):
    scores, labels = logits.softmax(-1)[:, 1:].max(-1)
    return labels + 1, scores.sort(descending=True).indices.argsort() + 1

def gc_hit(logits, graph, row):
    outcome = evaluate_reference_image(
        logits, graph.edge_label_index, graph.gt_relation_index[:, row:row+1],
        graph.gt_predicate[row:row+1], graph.boxes, graph.object_labels, ks=(50,))
    return bool(outcome['hits'][50])

cases = {True: [], False: []}
seen = {True: set(), False: set()}
with torch.inference_mode():
    for graph_index, image_id in enumerate(graphs.ids):
        graph = graphs[graph_index]
        if not graph.gt_predicate.numel():
            continue
        batch = collate([graph], dims['num_classes'])
        logits = aligng(**model_inputs(batch, 'cpu'))
        labels, ranks = pair_predictions(logits)
        indices = {tuple(pair): i for i, pair in enumerate(graph.edge_label_index.T.tolist())}
        chosen = {}
        for row, predicate in enumerate(graph.gt_predicate.tolist()):
            s, o = graph.gt_relation_index[:, row].tolist()
            pair = indices[s, o]
            pred, rank = int(labels[pair]), int(ranks[pair])
            hit = pred == predicate and rank <= 50
            if hit in chosen or len(cases[hit]) == 5 or predicate in seen[hit]:
                continue
            if gc_hit(logits, graph, row) != hit:
                continue
            chosen[hit] = (row, predicate, s, o, pair, pred, rank)
        if chosen:
            baseline = control(graph.x, graph.edge_label_index, graph.object_labels)
            ref_labels, ref_ranks = pair_predictions(baseline)
        for hit, (row, predicate, s, o, pair, pred, rank) in chosen.items():
            seen[hit].add(predicate)
            cases[hit].append(dict(image_id=image_id, graph_index=graph_index, gt_row=row,
                s=s, o=o, subject=objects[int(graph.object_labels[s])], object=objects[int(graph.object_labels[o])],
                gt=predicates[predicate], pred=predicates[pred], rank=rank,
                control=predicates[int(ref_labels[pair])], control_rank=int(ref_ranks[pair]),
                control_hit=gc_hit(baseline, graph, row), boxes=graph.boxes[[s,o]].tolist()))
        if all(len(group) == 5 for group in cases.values()):
            break
assert all(len(group) == 5 for group in cases.values())
for hit, group in cases.items():
    print('Hit' if hit else 'Miss')
    display(pd.DataFrame(group)[['image_id', 'subject', 'object', 'gt', 'control', 'control_rank', 'pred', 'rank']]
            .set_axis(['Ảnh', 'Chủ thể', 'Đối tượng', 'GT', 'PE-PLPR', 'Hạng PE', 'AlignG', 'Hạng AlignG'], axis=1))

Trên ảnh 1037, cùng cặp `pan → dough` có hai chú thích. GC chọn `to the left of`, vì vậy một chú thích là Hit và chú thích `of` là Miss. Trên ảnh 1060, hai mô hình cùng chọn đúng nhãn ở hạng 63.

## 2B. MagicBrush · Xóa ghế theo yêu cầu

Checkpoint nhánh prompt, seed 123, nhận đồ thị nguồn và câu lệnh. Sau khi suy luận, đối chiếu toàn bộ thao tác và đồ thị đích với đáp án test. Đây là 1/59 ca khớp hoàn toàn của checkpoint.

In [ ]:
from experiments.magicbrush_prompt_intent_v1.dataset import rows
from experiments.magicbrush_prompt_pilot_v1.run import example
from tools.emnlp2020_graph_modification_v1.dataset import sha256

chair = next(row for row in rows("test") if row["id"] == "magicbrush:190052:2")

def show_magicbrush_image(role):
    checksum = chair["images"][role]
    path = ROOT / "data/processed/level2/magicbrush-v1/images" / f"{checksum}.png"
    if sha256(path) != checksum:
        raise ValueError(f"Ảnh sai checksum: {role}")
    display(Image(filename=str(path), width=460))

print(chair["id"], chair["instruction"], sep="\n")
show_magicbrush_image("source_image")
chairs_source = EditGraph.from_dict(chair["source_graph"])
display(HTML(graph_html(chairs_source.to_dict(), "Đồ thị nguồn MagicBrush", "chair-source")))

In [ ]:
chair_ops, chair_graph, chair_checkpoint = predict(chairs_source, chair["instruction"], seed=123)
print("Checkpoint:", chair_checkpoint["checkpoint"])
print("Thao tác GNN:", json.dumps([op.to_dict() for op in chair_ops], ensure_ascii=False, indent=2))
display(HTML(graph_html(chair_graph.to_dict(), "Đồ thị sau khi áp dụng dự đoán", "chair-pred")))

In [ ]:
reference = example(chair, chair["operations"])
print("Khớp toàn bộ thao tác:", set(chair_ops) == set(reference.operations))
print("Khớp đồ thị đích:", chair_graph == reference.target)
print("Ảnh đích tham chiếu của MagicBrush")
show_magicbrush_image("target_image")

## 2B mở rộng. He và cộng sự · Chỉnh sửa điều kiện tìm ảnh

Ô này nạp hai dự đoán GNN seed 42 đã lưu (một đúng, một sai), rồi áp dụng thao tác lên đồ thị nguồn trên CPU. Đọc câu lệnh, so đồ thị dựng được với đồ thị tham chiếu. Một liên kết vô hướng trong biểu diễn này được mã hóa bằng hai cung đối xứng.

In [ ]:
root, manifest = open_bundle(ROOT)
for case_index, he_case in enumerate(task2_demo(root, manifest)):
    display(Markdown(f"### {he_case['id']} — exact theo evaluator đã lưu: {he_case['correct']}"))
    print("Instruction:", he_case["instruction"])
    print("Checkpoint SHA256:", he_case["checkpoint_sha256"])
    print("Predicted operations:")
    print(json.dumps(he_case["predicted"], ensure_ascii=False, indent=2))
    graphs = [("Nguồn Task 2", he_case["source"]),
              ("Đích dựng từ dự đoán", he_case["predicted_target"]),
              ("Đích tham chiếu", he_case["target"])]
    blocks = [f'<div style="flex:1;min-width:280px">{graph_html(g,title,f"he-{case_index}-{i}")}</div>'
              for i,(title,g) in enumerate(graphs)]
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:15px">'+''.join(blocks)+'</div>'))

## 3. VN100 · Ảnh sau chỉnh sửa có đúng yêu cầu và giữ được phần còn lại?

Yêu cầu là thêm chó trên đá ở tiền cảnh cạnh hồ. Qwen tạo ảnh từ ảnh nguồn và câu lệnh; REACT++ trích đồ thị riêng cho từng ảnh. Ô dưới đọc các kết quả đã lưu, kiểm tra SHA-256 và tính lại điểm nhất quán trên CPU. Khung đỏ đánh dấu đối tượng phát hiện được, khung vàng là vùng chỉnh sửa dự kiến.

In [ ]:
dog = recompute(root, manifest, 'travel-a4fe5a686b0396-q200-2')
print(dog['instruction'])
display(HTML(image_pair_html(root, dog)))
for role, title in [('source', 'Đồ thị ảnh nguồn'), ('target', 'Đồ thị ảnh Qwen')]:
    display(HTML(graph_html(dog['graphs_loaded'][role], title, 'vn-' + role)))
s, hand = dog['recomputed'], dog['scores_and_labels']
display(HTML(table(['E · chỉnh sửa', 'P · bảo toàn', 'G = E × P', 'Nhãn tay · chỉnh sửa', 'Nhãn tay · bảo toàn'],
                   [[s['edit_score'], round(s['preservation_score'], 3), round(s['graph_consistency'], 3),
                     hand['edit_fulfillment'], hand['preservation']]])))
print('Điều kiện đã kiểm tra:', s['checks'])

Bộ chấm phát hiện một đối tượng thuộc lớp chó mới trong vùng yêu cầu (E = 1). Các nút và cạnh được bảo toàn một phần (P ≈ 0,829), nên G ≈ 0,829. Người duyệt đánh dấu đạt cả hai tiêu chí. Điểm E ở đây kiểm tra lớp và vùng, còn chi tiết tư thế đứng trên đá cần xem trực tiếp trên ảnh.